In [1]:
import gymnasium as gym 
import random 
from pathlib import Path
import matplotlib.pyplot as plt 
import numpy as np
from tqdm import tqdm 
import pandas as pd  

In [2]:
class taxi_agent :
    def __init__(
            self,
            seed :int ,
            learning_rate:float,
            discount_factor:float,
            initial_epsilon: float , 
            final_epsilon : float , 
            env = gym.make("Taxi-v3"),
     
    ):
        self.env = env
        self.lr = learning_rate
        self.dis_f = discount_factor
        self.epsilon = initial_epsilon
        self.final_epsilon = final_epsilon

        self.epsilon_decay = 0.0

        self.n_states = env.observation_space.n
        self.n_actions = env.action_space.n 
        self.q_table = np.zeros((self.n_states,self.n_actions))

        self.training_error = []
        
        self.env.reset(seed=seed)
        np.random.seed(seed)
        random.seed(seed)
    
    def get_action (self, state , info):
        action_mask = info["action_mask"] 
        valid_actions = np.nonzero(action_mask == 1)[0]
        if np.random.random() < self.epsilon:
            action = np.random.choice(valid_actions)

        else:
            action = valid_actions[np.argmax(self.q_table[state,valid_actions] )]
        
        return action
    
    
    def update(self,
               state:int,
               action:int,
               next_info , 
               reward,
               terminated:bool,
               next_state:int
                ):
        action_mask = next_info["action_mask"] 
        next_valid_actions = np.nonzero(action_mask == 1)[0]
        
        future_q_values =  np.max(self.q_table[next_state,next_valid_actions])
        target = reward + self.dis_f*future_q_values*(not terminated)
        diff = target - self.q_table[state,action]

        self.q_table[state , action ] = self.q_table[state,action] + self.lr*diff
        self.training_error.append(diff)

    def decay_epsilon(self):
        self . epsilon = max(self.final_epsilon , self.epsilon - self.epsilon_decay)

    def train(self , n_episodes):
        self.epsilon_decay = self.epsilon / (n_episodes/2)
        for episode in tqdm(range(n_episodes)):
            state , info = self.env.reset()
            done  = False

            while not done :
                action = self.get_action(state,info)

                next_state, reward , terminated , truncated , next_info = self.env.step(action)
                
                self.update(
                    state,
                    action,
                    next_info , 
                    reward,
                    terminated,
                    next_state,)
                
                state = next_state
                info = next_info

                done = terminated or truncated
            self.decay_epsilon()
        self.env.close()
    
    def save_q_table(self,file_name: str = "taxi_q_table.xlsx"):
        df = pd.DataFrame(self.q_table)
        df.index.name = "State"
        df.columns = [f"Action_{i}" for i in range(self.q_table.shape[1])]
        df.to_excel(file_name)
        print(f"Successfully saved Q-table to {file_name}")
    
    def load_q_table(self,file_name="taxi_q_table.xlsx"):
        df = pd.read_excel(file_name, index_col=0)
        self.epsilon = 0.001

        q_table = df.to_numpy()
        print(f"Successfully loaded Q-table from {file_name}")
        self.q_table =  q_table
            







In [11]:
episodes = 5000

seed  = 100100
learning_rate = 0.1
discount_factor = 0.95
initial_epsilon = 0.95
final_epsilon = 0.01 

agent = taxi_agent(
    seed=seed,
    learning_rate=learning_rate,
    discount_factor=discount_factor,
    initial_epsilon=initial_epsilon,
    final_epsilon=final_epsilon,
    )

agent.train(episodes)

100%|██████████| 5000/5000 [00:06<00:00, 811.83it/s] 


In [12]:
agent.load_q_table()

Successfully loaded Q-table from taxi_q_table.xlsx


In [13]:
env = gym.make("Taxi-v3",render_mode = 'human') 

for i in range(10):
    done = False
    state,info = env.reset()
    while not done:
        action = agent.get_action(state,info)
        state , reward , terminated , truncated , info = env.step(action)
        done = terminated or truncated

env.close()



KeyboardInterrupt: 